# 00. Сравнение Qwen3-4B и Phi-4-mini на development v3

Это пункт 2 нового цикла. Две модели отвечают на 32 одинаковых вопроса с двумя
системными промптами. Получится **64 ответа на модель**, всего 128.
Перед основным прогоном проверяются chat template, токены остановки,
конечность logits и повторяемость короткого greedy-ответа.

Начните в новой сессии **Colab T4 GPU**. В первой ячейке выбрана `qwen3_4b`.
После завершения скачайте архив и выполненный ноутбук. Затем завершите эту сессию,
откройте новую T4-сессию и установите `MODEL_KEY = "phi4_mini"`.

Параметры и ревизии записаны в `data/model_screening_v3/protocol.json`.
Полного прогона на T4 при подготовке ноутбука ещё не было. Фактический результат
этой проверки появится после выполнения. Возможность LoRA-обучения проверяется
в пункте 3 отдельным training smoke-тестом.

[Подробная инструкция](https://github.com/maria-igntv/lm-hw/blob/main/docs/model_screening_v3.md).


In [ ]:
MODEL_KEY = "qwen3_4b"  # Во второй новой T4-сессии выбрать "phi4_mini".

import importlib.metadata
import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path("/content/lm-hw-v3")
REPOSITORY = "https://github.com/maria-igntv/lm-hw.git"
if not ROOT.exists():
    subprocess.run(["git", "clone", REPOSITORY, str(ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
current_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
if "bootstrap_commit" in globals() and bootstrap_commit != current_commit:
    raise RuntimeError("Код репозитория обновился в работающей сессии. Перезапустите сеанс Python и выполните ячейки заново.")
bootstrap_commit = current_commit

protocol = json.loads((ROOT / "data/model_screening_v3/protocol.json").read_text(encoding="utf-8"))
if MODEL_KEY not in protocol["models"]:
    raise ValueError("MODEL_KEY должен быть qwen3_4b или phi4_mini")
requirements = [f"{package}=={version}" for package, version in protocol["packages"].items()]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *requirements], check=True)

for package, version in protocol["packages"].items():
    module_name = package.replace("-", "_")
    loaded = sys.modules.get(module_name)
    if loaded is not None and getattr(loaded, "__version__", version) != version:
        raise RuntimeError("Библиотека уже импортирована в другой версии. Перезапустите сеанс Python и выполните ячейки заново.")
    if importlib.metadata.version(package) != version:
        raise RuntimeError(f"Не установилась закреплённая версия {package}")

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
print("Выбрана модель", MODEL_KEY)
print("Зависимости установлены. PyTorch и CUDA будут записаны из этой сессии Colab.")


## 1. Вопросы, оборудование и условия прогона

Общий промпт просит краткий русский ответ. `household` задаёт поведение бытового
ассистента при риске, неопределённости и обычных безопасных задачах.
Оба промпта доступны в протоколе до генерации. Модель получает текст вопроса
и выбранный системный промпт. Критерии правильности используются при экспорте судье.

Для основного сравнения выбран общий sampling с temperature 0.7, top_p 0.8,
top_k 20 и фиксированным seed каждого вопроса. Лимит одинаковый, 768 новых токенов.
Один ответ на условие даёт первоначальное сравнение. При близких результатах
нужны повторы с другими заранее заданными seed.

Веса загружаются NF4 на GPU с FP16 и eager attention. Вторая модель запускается
в новой сессии. Номер устройства, версии, код и входные файлы попадут в архив.


In [ ]:
import hashlib
import platform
import runpy
import shutil
import time
import traceback
from datetime import datetime, timezone

import torch

from checks.screening_v3 import (
    export_artifacts,
    finish_reason,
    read_jsonl,
    sha256_file,
    technical_indicators,
    validate_records,
    write_json,
    write_jsonl,
)

if not torch.cuda.is_available():
    raise RuntimeError("В Colab выберите Среда выполнения → Сменить среду выполнения → T4 GPU.")
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False

development_dir = ROOT / "data/development_v3"
runpy.run_path(str(development_dir / "validate.py"), run_name="__main__")
questions = read_jsonl(development_dir / "questions.jsonl")
if sha256_file(development_dir / "questions.jsonl") != protocol["questions_sha256"]:
    raise ValueError("Вопросы отличаются от закреплённого протокола")
candidate = protocol["models"][MODEL_KEY]
prompts = protocol["prompts"]
generation_settings = protocol["generation"]
SEED = protocol["seed"]

tracked_files = [
    "data/model_screening_v3/protocol.json",
    "data/development_v3/questions.jsonl",
    "data/development_v3/rubric.jsonl",
    "data/development_v3/sources.json",
    "data/development_v3/manifest.json",
    "checks/screening_v3.py",
    "checks/autocheck.py",
    "checks/evaluation.py",
    "judge/judge_prompt_v3.md",
    "judge/judge_response_schema_v3.json",
    "notebooks/00_model_screening_v3.ipynb",
]
versions = {
    package: importlib.metadata.version(package)
    for package in ["torch", *protocol["packages"]]
}
hardware = {
    "gpu": torch.cuda.get_device_name(0),
    "gpu_capability": list(torch.cuda.get_device_capability(0)),
    "gpu_memory_gib": torch.cuda.get_device_properties(0).total_memory / 1024**3,
    "torch_cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "nvidia_smi": subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
        text=True,
    ).strip(),
}
signature = {
    "protocol_id": protocol["protocol_id"],
    "model_key": MODEL_KEY,
    "candidate": candidate,
    "file_sha256": {name: sha256_file(ROOT / name) for name in tracked_files},
    "versions": versions,
    "hardware": hardware,
}
signature_hash = hashlib.sha256(json.dumps(signature, sort_keys=True).encode("utf-8")).hexdigest()
RUN_ID = f"{MODEL_KEY}_t{generation_settings['max_new_tokens']}_{signature_hash[:12]}"
OUT_DIR = ROOT / "results/model_screening_v3" / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)
metadata_path = OUT_DIR / "run_metadata.json"
if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if metadata["signature"] != signature:
        raise ValueError("Условия существующего прогона отличаются от текущих")
else:
    if (OUT_DIR / "answers.jsonl").exists():
        raise ValueError("Найдены ответы без метаданных прогона")
    metadata = {
        "run_id": RUN_ID,
        "signature_sha256": signature_hash,
        "signature": signature,
        "repo_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
        "repo_tracked_changes": subprocess.check_output(["git", "diff", "--name-only", "HEAD"], text=True).splitlines(),
        "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "status": "prepared",
        "expected_answers": len(questions) * len(prompts),
    }
    write_json(metadata_path, metadata)

for name in tracked_files:
    target = OUT_DIR / "reproduction" / name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ROOT / name, target)
(OUT_DIR / "pip_freeze.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8"
)
if not (OUT_DIR / "errors.jsonl").exists():
    write_jsonl(OUT_DIR / "errors.jsonl", [])


def record_error(stage, error):
    errors = read_jsonl(OUT_DIR / "errors.jsonl")
    errors.append({
        "at_utc": datetime.now(timezone.utc).isoformat(),
        "stage": stage,
        "type": type(error).__name__,
        "message": str(error),
        "traceback": traceback.format_exc(),
    })
    write_jsonl(OUT_DIR / "errors.jsonl", errors)
    metadata["status"] = "failed"
    metadata["last_error_stage"] = stage
    write_json(metadata_path, metadata)


print(json.dumps(hardware, ensure_ascii=False, indent=2))
print("Модель", candidate["model_id"])
print("Ревизия", candidate["revision"])
print("Вопросов", len(questions), "промптов", len(prompts), "ожидается ответов", metadata["expected_answers"])
print("Результаты", OUT_DIR)


## 2. Загрузка закреплённой модели и проверка формата диалога

Используются нативные реализации Transformers и chat template своего токенизатора.
Qwen3-Instruct-2507 работает в non-thinking-режиме. У Phi для завершения ответа
важен `<|end|>`, который учтён в опубликованном `generation_config.json`.
Полный эффективный список EOS сохраняется и проверяется.

Эта загрузка может занять несколько минут. При повторном запуске ячейки в той же
сессии уже загруженная модель используется повторно.


In [ ]:
from bitsandbytes.nn import Linear4bit
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GenerationConfig, set_seed

try:
    if "model" in globals():
        if globals().get("loaded_model_identity") != (candidate["model_id"], candidate["revision"]):
            raise RuntimeError("В памяти другая модель. Для следующего кандидата откройте новую T4-сессию.")
    else:
        load_started = time.perf_counter()
        tokenizer = AutoTokenizer.from_pretrained(
            candidate["model_id"], revision=candidate["revision"], trust_remote_code=False
        )
        quantization_settings = dict(protocol["quantization"])
        quantization_settings["bnb_4bit_compute_dtype"] = torch.float16
        model = AutoModelForCausalLM.from_pretrained(
            candidate["model_id"],
            revision=candidate["revision"],
            trust_remote_code=False,
            quantization_config=BitsAndBytesConfig(**quantization_settings),
            device_map={"": 0},
            dtype=torch.float16,
            attn_implementation=protocol["attention"],
        )
        loaded_model_identity = (candidate["model_id"], candidate["revision"])
        metadata["model_load_seconds"] = time.perf_counter() - load_started

    model.eval()
    tokenizer.padding_side = "left"
    official_generation = model.generation_config.to_dict()
    eos_ids = official_generation["eos_token_id"]
    if isinstance(eos_ids, int):
        eos_ids = [eos_ids]
    if set(eos_ids) != set(candidate["required_eos_token_ids"]):
        raise ValueError(f"Неожиданный список EOS {eos_ids}")
    if tokenizer.eos_token_id not in eos_ids:
        raise ValueError("EOS токенизатора отсутствует в конфигурации завершения")
    if type(model).__name__ != candidate["architecture"]:
        raise ValueError(f"Неожиданный класс модели {type(model).__name__}")
    if model.config._attn_implementation != protocol["attention"]:
        raise ValueError("Attention отличается от протокола")
    quantized_layers = sum(isinstance(module, Linear4bit) for module in model.modules())
    if not quantized_layers or not getattr(model, "is_loaded_in_4bit", False):
        raise ValueError("Не подтверждена 4-битная загрузка")
    if any(parameter.device.type != "cuda" for parameter in model.parameters()):
        raise ValueError("Часть параметров оказалась вне GPU")

    effective_generation = GenerationConfig.from_dict({**official_generation, **generation_settings})
    effective_generation.return_dict_in_generate = False
    effective_generation.output_scores = False
    if effective_generation.pad_token_id is None:
        effective_generation.pad_token_id = tokenizer.pad_token_id
    if effective_generation.pad_token_id is None:
        effective_generation.pad_token_id = eos_ids[0]
    effective_generation.validate()

    messages = [
        {"role": "system", "content": prompts["general"]},
        {"role": "user", "content": protocol["technical_controls"][0]["question"]},
    ]
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    direct_ids = tokenizer.apply_chat_template(messages, tokenize=True, return_dict=False, add_generation_prompt=True)
    if not rendered.endswith(candidate["assistant_prefix"]):
        raise ValueError("В конце диалога отсутствует ожидаемое начало ответа ассистента")
    if direct_ids != tokenizer(rendered, add_special_tokens=False)["input_ids"]:
        raise ValueError("Расходятся два способа токенизации chat template")

    write_json(OUT_DIR / "model_config.json", model.config.to_dict())
    write_json(OUT_DIR / "official_generation_config.json", official_generation)
    write_json(OUT_DIR / "effective_generation_config.json", effective_generation.to_dict())
    write_json(OUT_DIR / "special_tokens.json", {
        "tokenizer_eos_token_id": tokenizer.eos_token_id,
        "effective_eos_token_ids": eos_ids,
        "eos_token_texts": tokenizer.convert_ids_to_tokens(eos_ids),
        "pad_token_id": effective_generation.pad_token_id,
        "special_tokens_map": tokenizer.special_tokens_map,
    })
    (OUT_DIR / "chat_template.jinja").write_text(tokenizer.chat_template, encoding="utf-8")
    (OUT_DIR / "control_rendered_prompt.txt").write_text(rendered, encoding="utf-8")
    metadata["quantized_linear_layers"] = quantized_layers
    metadata["model_memory_gib"] = model.get_memory_footprint() / 1024**3
    metadata["status"] = "loaded"
    write_json(metadata_path, metadata)
except (Exception, KeyboardInterrupt) as error:
    record_error("model_load_and_template", error)
    raise

print("Класс модели", type(model).__name__)
print("NF4-слоёв", quantized_layers)
print("EOS", eos_ids, tokenizer.convert_ids_to_tokens(eos_ids))
print("Память модели, GiB", round(metadata["model_memory_gib"], 3))
print("Шаблон первого диалога")
print(rendered)


## 3. Единая функция генерации

Сохраняются реальный вход, токены ответа, текст со специальными токенами,
причина остановки, время и пик занятой GPU-памяти. Видимый ответ получается
декодированием только новых токенов с удалением специальных маркеров.
Если последний токен является EOS на позиции 768, ответ считается завершённым.
Пустой или оборванный ответ также попадёт в результаты и оценивание.


In [ ]:
@torch.inference_mode()
def generate_answer(question, system_prompt, seed, settings):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda:0")
    input_length = inputs["input_ids"].shape[1]
    if input_length + settings["max_new_tokens"] > model.config.max_position_embeddings:
        raise ValueError("Диалог и ответ не помещаются в контекст модели")
    config = GenerationConfig.from_dict({**effective_generation.to_dict(), **settings})
    set_seed(seed)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    output = model.generate(**inputs, generation_config=config)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    generated_ids = output[0, input_length:].tolist()
    return {
        "answer": tokenizer.decode(generated_ids, skip_special_tokens=True),
        "raw_answer": tokenizer.decode(generated_ids, skip_special_tokens=False),
        "rendered_prompt": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True),
        "input_token_ids": inputs["input_ids"][0].tolist(),
        "input_tokens": input_length,
        "generated_token_ids": generated_ids,
        "generated_tokens": len(generated_ids),
        "eos_token_ids": eos_ids,
        "max_new_tokens": settings["max_new_tokens"],
        "finish_reason": finish_reason(generated_ids, eos_ids, settings["max_new_tokens"]),
        "seed": seed,
        "elapsed_seconds": elapsed,
        "peak_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3,
    }


## 4. Технические контроли RU / EN

Четыре коротких запроса выполняются greedy. Первый русский запрос повторяется
с тем же seed, сравниваются токены. Для английского контроля используется
английский перевод общего промпта. Английские ответы служат диагностике.

Конечные logits подтверждают численную работоспособность этого forward pass.
Посмотрите, выполняется ли простая инструкция, осмысленен ли расчёт и нет ли
неожиданного смешения языков. Содержательные ошибки сохраняются как наблюдения.


In [ ]:
technical = {"status": "started", "controls": []}
try:
    probe_inputs = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": prompts["general"]},
            {"role": "user", "content": protocol["technical_controls"][0]["question"]},
        ],
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda:0")
    with torch.inference_mode():
        probe_output = model(**probe_inputs, use_cache=False)
        last_logits = probe_output.logits[0, -1].float()
        technical["last_logits_finite"] = bool(torch.isfinite(last_logits).all().item())
        technical["last_logits_min"] = float(last_logits.min().item())
        technical["last_logits_max"] = float(last_logits.max().item())
    del probe_output, last_logits, probe_inputs
    write_json(OUT_DIR / "technical_smoke.json", technical)
    if not technical["last_logits_finite"]:
        raise FloatingPointError("В logits найдены NaN или Inf")

    greedy_settings = {
        **generation_settings,
        "do_sample": False,
        "temperature": 1.0,
        "top_p": 1.0,
        "top_k": 0,
        "min_p": 0.0,
    }
    for control in protocol["technical_controls"]:
        system_prompt = prompts["general"]
        if control["language"] == "en":
            system_prompt = "You are a helpful assistant. Reply briefly and to the point in English."
        settings = {**greedy_settings, "max_new_tokens": control["max_new_tokens"]}
        result = generate_answer(control["question"], system_prompt, SEED, settings)
        technical["controls"].append({**control, **result, "system_prompt": system_prompt})
        write_json(OUT_DIR / "technical_smoke.json", technical)
        print(control["id"], result["finish_reason"], result["generated_tokens"])
        print(result["answer"])
        print()

    first_control = protocol["technical_controls"][0]
    repeated = generate_answer(
        first_control["question"], prompts["general"], SEED,
        {**greedy_settings, "max_new_tokens": first_control["max_new_tokens"]},
    )
    technical["repeated_first_control"] = repeated
    technical["greedy_tokens_equal"] = repeated["generated_token_ids"] == technical["controls"][0]["generated_token_ids"]
    technical["empty_control_ids"] = [control["id"] for control in technical["controls"] if not control["answer"].strip()]
    technical["ru_controls_with_cjk"] = [
        control["id"] for control in technical["controls"]
        if control["language"] == "ru" and technical_indicators(control["question"], control["answer"])["cjk_characters"]
    ]
    technical["status"] = "complete"
    write_json(OUT_DIR / "technical_smoke.json", technical)
    metadata["status"] = "technical_controls_complete"
    write_json(metadata_path, metadata)
except (Exception, KeyboardInterrupt) as error:
    technical["status"] = "failed"
    write_json(OUT_DIR / "technical_smoke.json", technical)
    record_error("technical_controls", error)
    raise

print("Greedy-токены при повторе совпали", technical["greedy_tokens_equal"])
print("Logits конечны", technical["last_logits_finite"])
print("Пустые контроли", technical["empty_control_ids"])
print("Русские контроли с CJK", technical["ru_controls_with_cjk"])


## 5. Основные 64 ответа

Один вопрос задаётся отдельно с каждым из двух промптов. Seed вопроса одинаков
для двух промптов и двух моделей. Chat template и EOS относятся к своей модели.
Критерии правильности в эту генерацию не подаются.

После каждого ответа весь небольшой JSONL атомарно заменяется. Повторный запуск
ячейки в той же сессии продолжает с отсутствующих пар вопрос / промпт.
Если выполнение прервано или возникла ошибка, уже сохранённые ответы можно
скачать последними двумя ячейками. В новой пустой сессии Colab файлы прошлой
сессии нужно сохранить заранее.


In [ ]:
answers_path = OUT_DIR / "answers.jsonl"
records = read_jsonl(answers_path)
done = validate_records(questions, prompts, records)
for record in records:
    expected_seed = SEED + next(index for index, question in enumerate(questions, 1) if question["id"] == record["id"])
    if record["run_id"] != RUN_ID or record["seed"] != expected_seed:
        raise ValueError("Сохранённый ответ относится к другим условиям")

metadata["status"] = "generating"
write_json(metadata_path, metadata)
try:
    for index, question in enumerate(questions, 1):
        for prompt_id, system_prompt in prompts.items():
            key = (prompt_id, question["id"])
            if key in done:
                continue
            result = generate_answer(question["question"], system_prompt, SEED + index, generation_settings)
            record = {
                **question,
                **result,
                "prompt_id": prompt_id,
                "run_id": RUN_ID,
                "model_key": MODEL_KEY,
                "model_revision": candidate["revision"],
                "technical": technical_indicators(question["question"], result["answer"]),
            }
            records.append(record)
            write_jsonl(answers_path, records)
            done[key] = record
            print(f"[{len(records)}/{metadata['expected_answers']}] {question['id']} {prompt_id} | {result['generated_tokens']} токенов | {result['finish_reason']}")
    validate_records(questions, prompts, records, complete=True)
    metadata["status"] = "generation_complete"
    metadata["answers_completed"] = len(records)
    metadata["generation_completed_at_utc"] = datetime.now(timezone.utc).isoformat()
    write_json(metadata_path, metadata)
except (Exception, KeyboardInterrupt) as error:
    metadata["answers_completed"] = len(read_jsonl(answers_path))
    record_error("development_generation", error)
    raise

print("Сохранено ответов", len(records))


## 6. Сводка, тексты и задания судье

Автосводка показывает причины остановки, CJK, повторы и regex-маркеры.
Наличие маркера само по себе не устанавливает смысловую безопасность.
Для короткого правильного ответа отсутствует минимальный порог слов.

`judge_requests.jsonl` содержит 64 задания для того же Qwen3.8-27B. В каждом
вопрос, ответ, критерии и основания. Идентификаторы непрозрачны, названия
кандидата и промпта остаются в отдельной таблице соответствия.
Запуск судьи выполняется после получения архивов двух кандидатов.
Версия судейского протокола обновлена под критерии development v3.

Эту ячейку можно выполнить после прерванной генерации, чтобы собрать частичный
архив. Полнота сохраняется в `automatic_summary.json` и `run_metadata.json`.


In [ ]:
records = read_jsonl(OUT_DIR / "answers.jsonl")
rubrics = read_jsonl(development_dir / "rubric.jsonl")
sources = json.loads((development_dir / "sources.json").read_text(encoding="utf-8"))["sources"]
judge_system = (ROOT / protocol["judge"]["system_prompt_path"]).read_text(encoding="utf-8")
judge_schema = json.loads((ROOT / protocol["judge"]["response_schema_path"]).read_text(encoding="utf-8"))
exported = export_artifacts(
    OUT_DIR, questions, prompts, records, rubrics, sources,
    judge_system, judge_schema, RUN_ID, SEED,
)
write_json(OUT_DIR / "judge_run_config.json", {
    **protocol["judge"],
    "protocol_id": "judge_v3.0",
    "expected_requests": len(questions) * len(prompts),
    "exported_requests": exported,
    "response_format": {"type": "json_schema", "json_schema": {"name": "household_judge_v3", "strict": True, "schema": judge_schema}},
})
summary = json.loads((OUT_DIR / "automatic_summary.json").read_text(encoding="utf-8"))
metadata["answers_completed"] = len(records)
metadata["complete"] = summary["complete"]
technical_path = OUT_DIR / "technical_smoke.json"
metadata["technical_status"] = "not_run"
if technical_path.exists():
    metadata["technical_status"] = json.loads(technical_path.read_text(encoding="utf-8"))["status"]
metadata["status"] = "partial_export"
if summary["complete"]:
    metadata["status"] = "complete" if metadata["technical_status"] == "complete" else "answers_complete_check_technical"
metadata["exported_at_utc"] = datetime.now(timezone.utc).isoformat()
write_json(metadata_path, metadata)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Заданий судье", exported)
print("Тексты всех ответов", OUT_DIR / "answers.md")

for question_id in ["dev3_01", "dev3_07", "dev3_26"]:
    for record in records:
        if record["id"] == question_id:
            print("=" * 80)
            print(record["id"], record["prompt_id"], record["finish_reason"])
            print(record["question"])
            print(record["answer"])


## 7. Скачать результат

Скачайте ZIP этой ячейкой. Затем выберите **Файл → Скачать → Скачать файл IPYNB**,
чтобы сохранить вывод выполненного ноутбука.

После первого архива завершите сессию и повторите notebook на T4
с `MODEL_KEY = "phi4_mini"`. Полученные два ZIP и два выполненных IPYNB
передайте для анализа. Для выбора модели будут сопоставлены технические проверки,
содержательные оценки и примеры ошибок. Переход к обучению требует результата
этого сравнения и отдельного T4 training smoke-теста из пункта 3.


In [ ]:
from google.colab import files

archive = shutil.make_archive(
    str(Path("/content") / ("screening_v3_" + RUN_ID)),
    "zip",
    root_dir=OUT_DIR.parent,
    base_dir=OUT_DIR.name,
)
files.download(archive)
print("Скачивается", archive)
print("Сохранено ответов", len(read_jsonl(OUT_DIR / "answers.jsonl")), "из", len(questions) * len(prompts))
